In [ ]:
import re
import random
import string
import time
import os
from tqdm import tqdm
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

df_train.shape, df_test.shape

In [ ]:
df_train.head()

In [ ]:
df_train["target"].value_counts()

In [ ]:
plt.bar(["0", "1"], df_train["target"].value_counts().values)

In [ ]:
keyword_null = df_train["keyword"].isnull().sum()
location_null = df_train["location"].isnull().sum()

print(f"Keyword: {100 * keyword_null / df_train.shape[0]:.2f}% null values")
print(f"Location: {100 * location_null / df_train.shape[0]:.2f}% null values")

In [ ]:
df_train = df_train.drop(columns=["location"], axis=1)
df_test = df_test.drop(columns=["location"], axis=1)

df_train.head()

In [ ]:
df_train["keyword"].unique()

In [ ]:
def clean_keyword(keyword):
    keyword = keyword.fillna("")
    keyword = keyword.str.replace("%20", " ") 
    keyword = keyword.str.replace(r"\s+", " ", regex=True) # Remove extra whitespace
    return keyword


def remove_url(text):
    url = re.compile(r'https?://\S+|www\.\S+')
    return url.sub(r'', text)


def remove_emoji(text):
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'  # emoticons
        u'\U0001F300-\U0001F5FF'  # symbols & pictographs
        u'\U0001F680-\U0001F6FF'  # transport & map symbols
        u'\U0001F1E0-\U0001F1FF'  # flags (iOS)
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)


def remove_html(text):
    html = re.compile(r'<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
    return re.sub(html, '', text)


# Special thanks to https://www.kaggle.com/tanulsingh077 for this function
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub(
        'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', 
        '', 
        text
    )
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    
    text = remove_url(text)
    text = remove_emoji(text)
    text = remove_html(text)
    
    return text

In [ ]:
df_train["keyword"] = clean_keyword(df_train["keyword"])
df_test["keyword"] = clean_keyword(df_test["keyword"])

df_train["text"] = df_train["text"].apply(clean_text)
df_test["text"] = df_test["text"].apply(clean_text)

df_train.iloc[80:90]

In [ ]:
df_train[df_train.duplicated(subset=["text", "target"])]

In [ ]:
# Drop duplicated text with the same labels
df_train = df_train.drop_duplicates(subset=["text", "target"], keep="first")

In [ ]:
df_train[df_train.duplicated("text")]

- **We also need to drop duplicated text with different labels**

In [ ]:
text_str = df_train[df_train["id"] == 356]["text"].values[0]
df_train[df_train["text"] == text_str]

In [ ]:
df_train = df_train.drop_duplicates("text", keep=False)
df_train[df_train.duplicated("text")]

In [ ]:
df_train = df_train.reset_index(drop=True)
df_train

In [ ]:
# Combine keyword with the text
def concatenate_text(df):
    for i, row in df.iterrows():
        if row["keyword"] != "":
            df.loc[i, "text"] = row["keyword"] + ": " + row["text"]
    df = df.drop(columns=["keyword"], axis=1)
    return df

In [ ]:
df_train = concatenate_text(df_train)
df_test = concatenate_text(df_test)

In [ ]:
df_train.iloc[100:115]

# GloVe

In [ ]:
train_text = df_train["text"].values
test_text = df_test["text"].values
train_label = df_train["target"].values

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_text)

vocab_length = len(tokenizer.word_index) + 1
vocab_length

In [ ]:
embeddings_index = {}

with open("/kaggle/input/glove840b300dtxt/glove.840B.300d.txt", "r", encoding="utf-8") as f:
    for line in tqdm(f):
        records = line.split(" ")
        word = records[0]
        coefs = np.asarray(records[1:], dtype="float32")
        embeddings_index[word] = coefs

In [ ]:
embeddings_matrix = np.zeros((vocab_length, 300))

for word, idx in tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embeddings_matrix[idx] = embedding_vector

embeddings_matrix

In [ ]:
def embed(texts):
    return tokenizer.texts_to_sequences(texts)

In [ ]:
longest_train_text = max(train_text, key=lambda x: len(embed([x])[0]))
length_longest_text = len(embed([longest_train_text])[0])

length_longest_text

In [ ]:
train_padded_text = pad_sequences(
    embed(train_text),
    length_longest_text,
    padding="post"
)

test_padded_text = pad_sequences(
    embed(test_text),
    length_longest_text,
    padding="post"
)

train_padded_text

In [ ]:
train_padded_text = torch.from_numpy(train_padded_text)
test_padded_text = torch.from_numpy(test_padded_text)
train_label = torch.from_numpy(train_label)
embeddings_matrix = torch.from_numpy(embeddings_matrix)

# LSTM

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(train_padded_text, train_label, test_size=0.2, random_state=42)

In [ ]:
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
val_dataset = torch.utils.data.TensorDataset(X_val, y_val)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
dataloaders = {
    "train": train_loader,
    "val": val_loader
}

In [ ]:
class GloVeLSTM(nn.Module):
    
    def __init__(self, hidden_size, num_layers, dropout_rate):
        super().__init__()
        self.embeddings = nn.Embedding.from_pretrained(embeddings_matrix.float(), freeze=True)
        self.lstm = nn.LSTM(
            input_size=300,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.norm = nn.BatchNorm1d(hidden_size * 2)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 2)

    def forward(self, x):
        embedded = self.embeddings(x)
        
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out.permute(0, 2, 1)
        
        pooled_out = F.max_pool1d(lstm_out, kernel_size=lstm_out.shape[2]).squeeze(2)
        
        bn_out = self.norm(pooled_out)
        bn_out = self.dropout(bn_out)
        
        fc1_out = F.relu(self.fc1(bn_out))
        fc1_out = self.dropout(fc1_out)

        fc2_out = F.relu(self.fc2(fc1_out))
        fc2_out = self.dropout(fc2_out)

        final_out = self.fc3(fc2_out)

        return final_out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

In [ ]:
model = GloVeLSTM(300, 2, 0.5)
model = model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
# num_epochs=5
# for epoch in range(num_epochs):
#     model.train()
#     train_loss = 0.0
#     for inputs, labels in train_loader:
#         inputs, labels = inputs.to(device), labels.to(device)
#         outputs = model(inputs)

#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         optimizer.zero_grad()

#         train_loss += loss.item()

#     model.eval()
#     with torch.no_grad():
#         val_loss = 0.0
#         correct_examples = 0
#         for inputs, labels in val_loader:
#             inputs, labels = inputs.to(device), labels.to(device)
#             outputs = model(inputs)

#             loss = criterion(outputs, labels)

#             val_loss += loss.item()
#             predictions = torch.argmax(outputs, 1)
#             correct_examples += (predictions == labels).sum().item()
        
#     train_loss /= len(train_loader)
#     val_loss /= len(val_loader)
    
#     print(
#         f"Epoch {epoch+1}: Training loss - {train_loss:.4f}, Validation loss - {val_loss:.4f}, "
#         f"Accuracy - {100 * correct_examples / len(val_loader.dataset):.2f}%"
#     )

In [ ]:
def train_model(model, optimizer, criterion, num_epochs=5):
    start_time = time.time()

    with TemporaryDirectory() as tempdir:
        best_model_params_path = os.path.join(tempdir, "best_model_params.pt")

        torch.save(model.state_dict(), best_model_params_path)
        best_acc = 0.0

        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}/{num_epochs}")
            print("-" * 10)
            
            for phase in ["train", "val"]:
                if phase == "train":
                    model.train()
                else:
                    model.eval()

                epoch_loss = 0.0
                epoch_acc = 0
                correct_examples = 0
                for inputs, labels in dataloaders[phase]:
                    inputs, labels = inputs.to(device), labels.to(device)

                    optimizer.zero_grad()
                    with torch.set_grad_enabled(phase == "train"):
                        outputs = model(inputs)

                        loss = criterion(outputs, labels)
                        predictions = torch.argmax(outputs, 1)

                        if phase == "train":
                            loss.backward()
                            optimizer.step()

                    epoch_loss += loss.item()
                    correct_examples += (predictions == labels).sum().item()

                epoch_loss /= len(dataloaders[phase])
                epoch_acc = correct_examples / len(dataloaders[phase].dataset)

                print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} - Acc: {100 * epoch_acc:.2f}%")

                if phase == "val" and epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), best_model_params_path)

            print("")

        print(f"Training completed in: {time.time() - start_time:.2f}s")
        print(f"Best val acc: {100 * best_acc:.2f}%")

        model.load_state_dict(torch.load(best_model_params_path, weights_only=True))

    return model

In [ ]:
trained_model = train_model(model, optimizer, criterion, 10)

In [ ]:
test_loader = torch.utils.data.DataLoader(test_padded_text, batch_size=8, shuffle=False)

In [ ]:
def predict(model, dataloader):
    model.eval()
    predictions = np.array([])
    with torch.inference_mode():
        for texts in dataloader:
            texts = texts.to(device)
            outputs = model(texts)
            y_preds = torch.argmax(outputs, 1)
            y_preds = y_preds.cpu().numpy()
            predictions = np.append(predictions, y_preds)
    return predictions.flatten().astype(int).tolist()

In [ ]:
predictions = predict(trained_model, test_loader)

In [ ]:
submission = pd.DataFrame({
    "id": df_test["id"],
    "target": predictions
})

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)
print("Submitted successfully!")